In [1]:
import tensorly as tl

tl.get_backend()

'numpy'

In [2]:
from hoda.hoda import HODA
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from mne.decoding import Scaler
import numpy as np
from sklearn.pipeline import Pipeline
from hoda.hoda import AutoBTTDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


hoda_params = dict(
    max_iter=128,
    tol=1e-8,
    init ='svd',
    shrinkage='lw',
    toeplitz=None,
    obj='tr',
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=True,
)

bttda = AutoBTTDA(hoda_params=hoda_params, verbose=True, forward=False, extra_train_info=False)


clf = Pipeline([
    ('scaler1', Scaler(scalings='mean', with_mean=True)),
    ('bttda', bttda),
    ('vec', FunctionTransformer(tl.unfold, kw_args=dict(mode=0))),
    ('numpy', FunctionTransformer(tl.to_numpy)),
    ('scaler2', StandardScaler()),
    ('clf', LDA(shrinkage='auto', solver='lsqr'))
])
deltas = [0] + list(np.geomspace(1e-3,1, 5-1))
clf

Pipeline(steps=[('scaler1',
                 Scaler({'info': None, 'scalings': 'mean', 'with_mean': True, 'with_std': True})),
                ('bttda',
                 AutoBTTDA(forward=False,
                           hoda_params={'extra_train_info': False,
                                        'init': 'svd', 'max_iter': 128,
                                        'obj': 'tr', 'shrinkage': 'lw',
                                        'solver': 'lanczos', 'taper': False,
                                        'toeplitz': None, 'tol': 1e-08,
                                        'verbose': True},
                           verbose=True)),
                ('vec',
                 FunctionTransformer(func=<function unfold at 0x7f3c2b00f6a0>,
                                     kw_args={'mode': 0})),
                ('numpy',
                 FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x7f3c2b00f240>)),
                ('scaler2', StandardScaler()),
                ('clf',
                 LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))])

In [3]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
from sklearn.model_selection import StratifiedKFold, GridSearchCV

sfreq = 48
paradigm = P300(resample=sfreq)
datasets = [BNCI2014_008()]

param_grid = dict(
    bttda__delta = [0, 1e-4, 1e-3, 1e-2, 1e-1,  0.5],
    bttda__n_blocks = [ 1, 2,3,4,5,6,7,8,9,10],

)
cv = StratifiedKFold(n_splits=5)


In [4]:
def melt_cv_results(results_df):
    # Identify all split columns for test and train scores
    split_columns = [col for col in results_df.columns if col.startswith("split") and ("test_score" in col or "train_score" in col)]
    
    # Melt the DataFrame to bring split columns into rows
    melted_df = results_df.melt(
        id_vars=[col for col in results_df.columns if col not in split_columns],
        value_vars=split_columns,
        var_name="split_score",
        value_name="score"
    )
    
    # Extract split (fold) and score type (test/train) into separate columns
    melted_df["fold"] = melted_df["split_score"].str.extract(r'split(\d+)_')[0].astype(int)
    melted_df["score_type"] = melted_df["split_score"].str.extract(r'_(test_score|train_score)')[0]
    
    # Drop the original split_score column for clarity
    melted_df = melted_df.drop(columns=["split_score"])
    
    return melted_df

In [8]:
import pandas as pd
import tensorly as tl
from sklearn import config_context


results = []

gs = GridSearchCV(
    clf,
    param_grid,
    scoring='roc_auc',
    n_jobs=1,
    refit=False,
    cv=cv,
    verbose=True,
    return_train_score=True
)

for dataset in datasets:
    for subject in dataset.subject_list[:1]:
        X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[subject])
        X = tl.tensor(X)
        gs.fit(X,labels)
        res = pd.DataFrame(gs.cv_results_)
        res = melt_cv_results(res)
        res["dataset"] = dataset.code
        res["subject"] = subject
        results.append(res)
        
results = pd.concat(results, ignore_index=True)

/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fitting block 1/1...


Backward HODA model rank=(8, 48):   1%|▍                                               | 1/128 [00:00<00:31,  4.01it/s]


Fitting block 1/1...


Backward HODA model rank=(8, 48):   1%|▍                                               | 1/128 [00:00<00:23,  5.46it/s]


...

Backward HODA model rank=(1, 2):  23%|███████████▎                                    | 30/128 [00:00<00:02, 35.66it/s]


Fitting block 1/2...


Forward model :  24%|███████████████▋                                                 | 31/128 [00:00<00:02, 37.26it/s]


Fitting block 2/2...


Backward HODA model rank=(2, 2):  23%|██████████▉                                     | 29/128 [00:01<00:04, 21.88it/s]

KeyboardInterrupt



In [ ]:
results

In [ ]:
import plotly.express as px
df = results.query("score_type == 'test_score'")
df = df.groupby(['score_type', 'param_bttda__n_blocks', 'param_bttda__delta'])['score'].aggregate('mean')
df = df.reset_index()
px.line(df, x='param_bttda__n_blocks',y='score', color='param_bttda__delta', color_discrete_sequence=px.colors.sequential.Reds[3:])
